# Ferrozine Analysis
This notebooks is written to analyze data from a ferrozine analysis. The input data will need to be properly formatted to be read by this script.

# Input data and check
Replace the values below with your own data. Use comma separated lists for the data.

In [ ]:
# import the numpy module so we can call various functions for the linear regression
import numpy as np
import pandas as pd

# iron standard concentrations in micromolar
stds = [10, 20, 30, 40]
# A1 absorbances
A1_abs = [0.002, 0.004, 0.006, 0.008]
# A2 absorbances
A2_abs = [0.124, 0.229, 0.325, 0.431]
# A2' absorbances
A2p_abs = [0.108, 0.201, 0.295, 0.390]
# Absorbances for you unknown sample use a comma to separate measurements
A1_abs_samples = [0.352]
A2_abs_samples = [0.464]
# name of samples
location = ["test sample"]

# Parameter derivation
Now that the data is uploaded we need to derive some values for the analysis. The parameters we need to derive from the data are $\epsilon_{Fe(II)}$, $\epsilon_{Fe(III)}$, and $\alpha$. The pathlength ($l$) is 1 cm and $A_1$ and $A_2$ are the absorbance measurments you are given. 

$$C_{Fe(II)}=\frac{A_1\epsilon{_{Fe(II)}}l\alpha-A_2\epsilon{_{Fe(III)}}l}{\epsilon{_{Fe(II)}}l\alpha(\epsilon{_{Fe(II)}}l-\epsilon{_{Fe(III)}}l}$$

$$C_{Fe(III)}=\frac{A_2-A_1\alpha}{\alpha(\epsilon{_{Fe(II)}}l-\epsilon{_{Fe(III)}}l)}$$

Let's start with deriving $\epsilon_{Fe(III)}$ from the $A_1$ measurements of the standards. We need to perform a linear regression with the $A_1$ measurements and iron concentrations of the standards.

In [ ]:
# get the coefficients for the linear regression fit to our extracted portion of data
A1coefficients = np.polyfit(stds, A1_abs, 1)
epsFe3 = A1coefficients[0]
# put those coefficients into a 1st degree polynomial so we can calculate some points to
# show the linear regression on our plot
A1polynomial = np.poly1d(A1coefficients)
A1fit_line = A1polynomial(stds)

# calculate the coefficient of determination also known as r^2
residuals = A1_abs - A1fit_line
ss_res = np.sum(residuals**2)
ss_tot = np.sum((A1_abs - np.mean(A1_abs)) ** 2)
r_squared = 1 - (ss_res / ss_tot)
print("r squared = %.3f" % r_squared)
print("Epsilon Fe(III) = %g" % epsFe3)

# to plot the data we need to import matlibplot
import matplotlib.pyplot as plt

# to make the plots look nicer we will import the seaborn module
import seaborn as sns

sns.set_theme()
sns.set_context("paper")
# plot the data and label the axes
plt.figure()
plt.scatter(stds, A1_abs)
plt.plot(stds, A1fit_line)
plt.xlabel(r"Fe(III) $\mu$M")
plt.ylabel("A1 absorbance")

# Save and download the plot in various file formats
plt.savefig("ferrozine_A1.png", bbox_inches="tight")
plt.savefig("ferrozine_A1.pdf", bbox_inches="tight")

# $\epsilon_{Fe(II)}$ derivation
Now let's calculate $\epsilon_{Fe(II)}$ from the $A_2$ measurements of the standards.

In [ ]:
# get the coefficients for the linear regression fit to our extracted portion of data
A2coefficients = np.polyfit(stds, A2_abs, 1)
epsFe2 = A2coefficients[0]

# put those coefficients into a 1st degree polynomial so we can calculate some points to
# show the linear regression on our plot
A2polynomial = np.poly1d(A2coefficients)
A2fit_line = A2polynomial(stds)

# calculate the coefficient of determination also known as r^2
residuals = A2_abs - A2fit_line
ss_res = np.sum(residuals**2)
ss_tot = np.sum((A2_abs - np.mean(A2_abs)) ** 2)
r_squared = 1 - (ss_res / ss_tot)
print("r squared = %.3f" % r_squared)
print("Epsilon Fe(II) = %g" % epsFe2)

# plot the data and label the axes
plt.figure()
plt.scatter(stds, A2_abs)
plt.plot(stds, A2fit_line)
plt.xlabel(r"Fe(II) $\mu$M")
plt.ylabel("A2 absorbance")

# Save and download the plot in various file formats
plt.savefig("ferrozine_A2.png", bbox_inches="tight")
plt.savefig("ferrozine_A2.pdf", bbox_inches="tight")

# Dilution factor
The last variable we need to derive from our data is dilution factor: $\alpha$.

In [ ]:
# to calculate the dilution factor we need to take the average dilution of our standards
alpha = np.mean(np.divide(np.array(A2p_abs), np.array(A2_abs)))
print("alpha = %.2f" % alpha)

# Calculation of iron in unknown samples
Now we can calculate Fe(III) and Fe(II) in our sample from the absorbance measurements. 

In [ ]:
for i in range(len(location)):
    if pd.isna(A1_abs_samples[i]) == False:
        # sample absorbance measurements
        A1 = A1_abs_samples[i]
        A2 = A2_abs_samples[i]

        # length of cuvette
        l = 1

        # Fe(III) concentration
        Fe3 = (A2 - A1 * alpha) / (alpha * (epsFe2 * l - epsFe3 * l))

        # Fe(II) concentration
        Fe2 = (A1 * epsFe2 * l * alpha - A2 * epsFe3 * l) / (
            epsFe2 * l * alpha * (epsFe2 * l - epsFe3 * l)
        )

        # print the concentrations
        print("%s: Fe(III) = %.2f uM, Fe(II) = %.2f uM" % (location[i], Fe3, Fe2))